# Stage 1 Drift Run — Single Model & Single Dataset

**What this notebook does:** for a sample of real MATH-500 problems, runs BOTH conditions, matching Mirage (2506.04210)'s TTBC1 exactly:
- **Short**: true natural generation, no injection, no artificial ceiling — the model's own unmodified stopping point.
- **Long**: Wait injection (literal "Wait"), UNCAPPED — every genuine stop attempt gets intercepted, with only the model's real max generation length (32768) as the ceiling.

Then checks whether the answer flips between the two, and in which direction:
- **drift**: correct short -> wrong long (the phenomenon we're testing for).
- **reverse**: wrong short -> correct long (the mirror; common reverse flips would suggest noise, not directional drift).
- **stable_correct** / **stable_wrong**: no flip.
- **short_truncated** / **long_truncated**: the model never reached a boxed answer in that condition (reported honestly, not dropped).

**Model is switchable in Section 2.5** — change one line (`MODEL_KEY`) to run DeepSeek-R1-Distill-Qwen-1.5B, -Qwen-7B, or -Llama-8B, or add a new one to the dict. Output filenames are tagged with the model automatically, so results never collide across models.

**This notebook auto-detects how many GPUs are actually available** and picks the right strategy: multi-GPU parallel workers if several are visible, or a single sequential run (with resume support) if only one is.

**Files to upload first:** `harness.py` (v0.3.4), `wait_injection.py`, `math500_matcher.py`, `drift_run_worker.py`. Run top to bottom.

## 1. Install packages
Let this finish completely before running anything else.

In [ ]:
!pip install -q numpy torch transformers accelerate datasets


## 2. Version check — FIRST, every time

In [ ]:
import harness, wait_injection, math500_matcher, inspect, os
print("harness version:", harness.HARNESS_VERSION, "(want 0.3.4)")
print("wait_injection version:", getattr(wait_injection, "WAIT_INJECTION_VERSION", "MISSING -- STALE FILE"), "(want 0.2.0)")
print("Wait default phrasing:", repr(wait_injection.WAIT_TEXT), "(want 'Wait')")

wi_src = inspect.getsource(wait_injection)
has_uncapped_fix = "max_injections is not None and" in wi_src
print("wait_injection has uncapped-injection fix:", has_uncapped_fix, "(want True)")
if not has_uncapped_fix:
    print("  !! STOP: this file will crash with 'TypeError: >= not supported between "
          "int and NoneType' the moment it hits a genuine stop attempt. Re-upload "
          "wait_injection.py before running anything else.")

src = inspect.getsource(math500_matcher)
print("matcher has x-in-prefix fix:", "_VAR_IN_PREFIX" in src, "(want True)")
print("matcher has nested-frac fix:", "_FRAC_BRACED" in src, "(want True)")

import torch
N_GPUS = torch.cuda.device_count()
print("GPUs visible to this notebook:", N_GPUS)
print("CUDA_VISIBLE_DEVICES (env):", os.environ.get("CUDA_VISIBLE_DEVICES", "(not set)"))
print()
if N_GPUS >= 2:
    print(f">> Will use the MULTI-GPU path (Section 5A) across {N_GPUS} GPUs.")
else:
    print(">> Will use the SINGLE-GPU sequential path (Section 5B), since only 1 GPU is visible.")


## 2.5. Choose the model

**To switch models, change ONE line: `MODEL_KEY` below.** Everything downstream (single-GPU and multi-GPU paths, output filenames) picks it up automatically, no other edits needed. Add a new model later by adding one line to `AVAILABLE_MODELS`.

Results are saved with the model name in the filename, so switching models never overwrites a previous model's results.

In [ ]:
AVAILABLE_MODELS = {
    "qwen-1.5b": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    "qwen-7b":   "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    "llama-8b":  "deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    # add more here, e.g. "qwen-14b": "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B",
}

MODEL_KEY = "llama-8b"   # <-- change this one line to switch models
MODEL_NAME = AVAILABLE_MODELS[MODEL_KEY]
print(f"Using model: {MODEL_KEY} -> {MODEL_NAME}")

# NOTE: this assumes the model uses the same <think>/</think> reasoning-tag
# convention as the DeepSeek-R1-Distill family (all three listed do). A
# genuinely different model family (e.g. Qwen3 later) would need the token
# handling in harness.py re-verified before trusting this -- nothing
# transfers automatically across model families (see ROADMAP.md, Rule M1).

## 3. Pick the sample

n=40, fixed seed, reproducible.

In [ ]:
import random
from datasets import load_dataset

ds = load_dataset("HuggingFaceH4/MATH-500", split="test")
N = 10   # smaller re-check run, now that the injection cap / budget fix is in
idxs = list(range(len(ds)))
random.Random(7).shuffle(idxs)   # same seed as the n=40 run, so these 10 are a
                                  # subset of what we already tried, direct comparison
all_idxs = idxs[:N]
print(f"selected {len(all_idxs)} problems for the drift run")


## 4. Smoke test (multi-GPU path only)

**Skip this cell entirely if Section 2 said to use the single-GPU path.** This is only useful when there are multiple GPUs to spread across.

In [ ]:
import subprocess, time

def launch(gpu_id, indices, out):
    log = open(f"{out}.log", "w")
    return subprocess.Popen(
        ["python", "drift_run_worker.py", "--gpu", str(gpu_id), "--device", str(gpu_id),
         "--indices", ",".join(map(str, indices)), "--model", MODEL_NAME,
         "--short_budget", "32768", "--long_budget", "32768", "--out", out],
        stdout=log, stderr=subprocess.STDOUT,
    ), log

if N_GPUS >= 2:
    test_procs = []
    for gpu_id, idx in zip([1, 2] if N_GPUS > 2 else [0, 1], all_idxs[:2]):
        p, log = launch(gpu_id, [idx], f"drift_test_gpu{gpu_id}_{MODEL_KEY}.json")
        test_procs.append((gpu_id, p, log))
        print(f"launched test on gpu {gpu_id} for idx {idx}")
    while any(p.poll() is None for _, p, _ in test_procs):
        time.sleep(15)
    for gpu_id, p, log in test_procs:
        print(f"gpu {gpu_id}: exit code {p.returncode}  (0 = good)")
    for gpu_id, _, _ in test_procs:
        print(f"\n--- drift_test_gpu{gpu_id}_{MODEL_KEY}.json.log ---")
        print(open(f"drift_test_gpu{gpu_id}_{MODEL_KEY}.json.log").read())
else:
    print("Only 1 GPU visible -- skipping the multi-GPU smoke test. Go to Section 5B.")


## 5A. Full launch — multi-GPU path

**Only run this if Section 2 detected 2+ GPUs and the Section 4 smoke test passed.** If you're on a single GPU, skip to 5B instead.

In [ ]:
if N_GPUS >= 2:
    WORKER_GPUS = list(range(1, N_GPUS)) if N_GPUS > 2 else [0, 1]
    shards = [all_idxs[i::len(WORKER_GPUS)] for i in range(len(WORKER_GPUS))]

    procs = []
    for gpu_id, shard in zip(WORKER_GPUS, shards):
        if not shard:
            continue
        p, log = launch(gpu_id, shard, f"drift_shard_gpu{gpu_id}_{MODEL_KEY}.json")
        procs.append((gpu_id, p, log))
        print(f"launched gpu {gpu_id} with {len(shard)} items")
    print(f"\n{len(procs)} workers running across GPUs {WORKER_GPUS}")
else:
    print("Only 1 GPU -- this cell does nothing. Use Section 5B instead.")


## 5A-check. Progress check (multi-GPU path only)

Safe to re-run any time. Skip if you're on the single-GPU path.

In [ ]:
if N_GPUS >= 2:
    done = sum(1 for _, p, _ in procs if p.poll() is not None)
    print(f"{done}/{len(procs)} workers finished")
    for gpu_id, p, _ in procs:
        print(f"  gpu {gpu_id}: {'done' if p.poll() is not None else 'running'}")
else:
    print("Only 1 GPU -- nothing to check here. See Section 5B's own progress prints.")


## 5B. Full run — single-GPU sequential path

**Only run this if Section 2 detected exactly 1 GPU.** Loads the model once in this notebook's own kernel (no subprocess needed, since there's only one GPU to use anyway) and works through all 40 items one at a time, saving after every item.

**This has resume support.** If this notebook (or the whole JupyterHub instance) dies partway through and you have to start a new one, re-upload `drift_run_results_singlegpu.json` alongside the other files before re-running this cell, and it will pick up where it left off instead of redoing finished items. If you don't have a previous partial file, it just starts fresh.

**This will take a while** — 40 items x 2 generations each, no parallelism. Expect this to run for a long stretch; that's normal for a single GPU, not a sign anything is wrong.

In [ ]:
import json, time
from transformers import AutoModelForCausalLM, AutoTokenizer
from harness import generate_natural
from wait_injection import generate_stop_triggered_wait
from math500_matcher import extract_math500_gold, extract_math500_pred, is_equiv_math
from drift_run_worker import categorize  # reuse the already-tested categorization logic

OUT_PATH = f"drift_run_results_singlegpu_{MODEL_KEY}.json"
# Matches Mirage (2506.04210)'s TTBC1 exactly: short = true natural generation
# (no artificial ceiling, the paper's own baseline), long = uncapped Wait
# injection, only the model's real max generation length (32768, DeepSeek's
# own documented limit) as the ceiling for either condition. The earlier
# SHORT_BUDGET=4096 / LONG_BUDGET=16384 / capped-at-4-injections version was
# our own invented design, not what either source paper describes, and
# likely explains the earlier 0-drift result on this model.
SHORT_BUDGET, LONG_BUDGET = 32768, 32768

if N_GPUS < 2:
    # resume support: load whatever's already been done for THIS model
    try:
        results = json.load(open(OUT_PATH))
        done_idxs = {r["math500_idx"] for r in results}
        print(f"Resuming ({MODEL_KEY}): {len(results)} items already done, skipping those.")
    except FileNotFoundError:
        results = []
        done_idxs = set()
        print(f"No previous partial results for {MODEL_KEY}, starting fresh.")

    print(f"Loading model {MODEL_NAME} (once, this kernel)...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda:0"
    )
    model.eval()
    print("Loaded. Starting sequential run...\n")
    print("NOTE: with uncapped injection and a 32768 ceiling, some items (especially")
    print("hard ones) may take a genuinely long time. That's expected, not a hang.\n")

    remaining = [i for i in all_idxs if i not in done_idxs]
    for rank, i in enumerate(remaining):
        item = ds[i]
        q = item["problem"]
        gold = extract_math500_gold(item)

        t0 = time.time()
        short_text, short_think, _, _, _ = generate_natural(
            model, tokenizer, q, max_new_tokens=SHORT_BUDGET, seed=0)
        dt_short = time.time() - t0
        short_pred = extract_math500_pred(short_text)
        short_correct, short_tier = is_equiv_math(short_pred, gold)
        short_closed = "</think>" in short_text  # informational; grading already
                                                   # handles the None (never-closed) case

        t0 = time.time()
        long_text, long_think, _, _, _, n_inj, inj_offsets = generate_stop_triggered_wait(
            model, tokenizer, q, long_budget=LONG_BUDGET, max_injections=None, seed=0)
        dt_long = time.time() - t0
        long_pred = extract_math500_pred(long_text)
        long_correct, long_tier = is_equiv_math(long_pred, gold)

        category = categorize(short_pred, short_correct, long_pred, long_correct)

        results.append(dict(
            math500_idx=i, unique_id=item["unique_id"], subject=item["subject"], level=item["level"],
            model=MODEL_KEY, gold=gold, category=category,
            short_pred=short_pred, short_correct=short_correct, short_tier=short_tier,
            short_think_tokens=short_think, short_closed_naturally=short_closed,
            short_wall_clock_sec=round(dt_short, 1), short_text=short_text,
            long_pred=long_pred, long_correct=long_correct, long_tier=long_tier,
            long_think_tokens=long_think, long_n_injections=n_inj,
            long_injection_offsets=inj_offsets, long_wall_clock_sec=round(dt_long, 1),
            long_text=long_text,
        ))
        with open(OUT_PATH, "w") as f:
            json.dump(results, f, indent=2)
        print(f"[{rank+1}/{len(remaining)}] idx={i} category={category} "
              f"short={short_correct} long={long_correct} n_inj={n_inj} "
              f"({dt_short:.0f}s + {dt_long:.0f}s)  [{len(results)}/{N} total done]")

    print(f"\nDone. {len(results)}/{N} items complete for {MODEL_KEY}, saved to {OUT_PATH}")
else:
    print("2+ GPUs visible -- use Section 5A instead, this cell does nothing.")


## 6. Merge and report — the actual result

Works with either path's output. **The key number is drift vs reverse.** Truncations are reported as their own honest category.

In [ ]:
import json
from collections import Counter

all_results = []
if N_GPUS >= 2:
    for gpu_id, p, log in procs:
        try:
            all_results.extend(json.load(open(f"drift_shard_gpu{gpu_id}_{MODEL_KEY}.json")))
        except (FileNotFoundError, json.JSONDecodeError):
            print(f"WARNING: no/partial output for gpu {gpu_id}, "
                  f"check drift_shard_gpu{gpu_id}_{MODEL_KEY}.json.log")
else:
    try:
        all_results = json.load(open(f"drift_run_results_singlegpu_{MODEL_KEY}.json"))
    except FileNotFoundError:
        print(f"No single-GPU results file found yet for {MODEL_KEY}.")

with open(f"drift_run_results_{MODEL_KEY}.json", "w") as f:
    json.dump(all_results, f, indent=2)

n = len(all_results)
if n == 0:
    print("No results collected yet.")
else:
    print(f"Model: {MODEL_KEY} ({MODEL_NAME})")
    cats = Counter(r["category"] for r in all_results)
    print(f"Total items: {n}\n")
    for cat in ["drift", "reverse", "stable_correct", "stable_wrong", "short_truncated", "long_truncated"]:
        print(f"  {cat:16s}: {cats.get(cat, 0)}")

    n_drift, n_reverse = cats.get("drift", 0), cats.get("reverse", 0)
    n_judged = n_drift + n_reverse + cats.get("stable_correct", 0) + cats.get("stable_wrong", 0)
    print(f"\nOf {n_judged} items where both conditions produced an answer:")
    if n_judged:
        print(f"  drift:   {n_drift} ({n_drift/n_judged:.1%})")
        print(f"  reverse: {n_reverse} ({n_reverse/n_judged:.1%})")
    n_trunc = cats.get('short_truncated', 0) + cats.get('long_truncated', 0)
    print(f"\nTruncated (reported, not discarded): {n_trunc}/{n} ({n_trunc/n:.1%})")

    if n_drift > n_reverse:
        print(f"\n>> drift ({n_drift}) outnumbers reverse ({n_reverse}): consistent with replication.")
    elif n_drift == n_reverse:
        print(f"\n>> drift and reverse are equal ({n_drift} each): no directional signal.")
    else:
        print(f"\n>> reverse ({n_reverse}) outnumbers drift ({n_drift}): unexpected, worth investigating.")


## 7. Hand-check ambiguous matcher calls

In [ ]:
to_check = [r for r in all_results
            if (r["short_tier"] == "no_match" and r["short_pred"] is not None)
            or (r["long_tier"] == "no_match" and r["long_pred"] is not None)]
print(f"{len(to_check)} items with a non-truncation no_match tier:\n")
for r in to_check:
    print(f"idx={r['math500_idx']} lvl={r['level']} category={r['category']} gold={r['gold']!r}")
    print(f"   short: pred={r['short_pred']!r} tier={r['short_tier']}")
    print(f"   long:  pred={r['long_pred']!r} tier={r['long_tier']}\n")


## 8. List the actual drift cases

In [ ]:
drift_cases = [r for r in all_results if r["category"] == "drift"]
print(f"{len(drift_cases)} drift cases:\n")
for r in drift_cases:
    print(f"idx={r['math500_idx']} lvl={r['level']} subj={r['subject']} gold={r['gold']!r}")
    print(f"   short (correct): {r['short_pred']!r}")
    print(f"   long (wrong):    {r['long_pred']!r}  ({r['long_n_injections']} injections)")
    print(f"   ...long tail: {r['long_text'][-300:]}\n")
